# Local Experts Pipeline — Radius Features (9 / 25 / 49 cells)

Duplicate of `local_experts_pipeline.ipynb` with enriched input features.

For every generation in the input chain, three neighbourhood-count channels are added
alongside the raw binary board:

| Radius | Window size | Cells counted |
|--------|-------------|---------------|
| r = 1  | 3 × 3       | 9             |
| r = 2  | 5 × 5       | 25            |
| r = 3  | 7 × 7       | 49            |

Each generation goes from 1 channel → 4 channels (binary + 3 counts).  
With GEN-1 = 3 generations the total input depth is **12 channels**.

**5 architectures evaluated**: MLP · CNN · RNN · CRNN · RCNN

---
## 0 · Imports & Configuration

In [ ]:
import importlib
import functions
import feature_engineering
importlib.reload(functions)
importlib.reload(feature_engineering)
from functions import *
from feature_engineering import add_radius_features

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt

print(f'TensorFlow {tf.__version__}')

In [ ]:
# ─── Global hyper-parameters ──────────────────────────────────────────────────
SIZE          = 10
AMOUNT_BOARDS = 1000
GEN           = 4
N_PIXELS      = SIZE * SIZE          # 100
TIMESTEPS     = GEN - 1              # 3 board snapshots used as input

RADII         = (1, 2, 3)            # → 9-cell, 25-cell, 49-cell windows
N_FEAT        = 1 + len(RADII)       # channels per generation: binary + 3 counts = 4

TEST_SIZE     = 0.10
VAL_SIZE      = 0.10
RANDOM_STATE  = 365

EPOCHS        = 50
BATCH_SIZE    = 256
ES_PATIENCE   = 5

# Separate root so these models never collide with the non-radius pipeline
MODEL_ROOT    = f'pixel_models_radius/size{SIZE}_gen{GEN}'
os.makedirs(MODEL_ROOT, exist_ok=True)

print(f'Radii          : {RADII}  →  {[( 2*r+1)**2 for r in RADII]}-cell windows')
print(f'Channels/gen   : {N_FEAT}  (1 binary + {len(RADII)} radius counts)')
print(f'Total channels : {TIMESTEPS * N_FEAT}  ({TIMESTEPS} gens × {N_FEAT})')
print(f'Model root     : {MODEL_ROOT}/')

---
## 1 · Load Raw Data & Build Radius Features

In [ ]:
reverse_df = load_reverse_df(SIZE, AMOUNT_BOARDS, GEN)
print(f'Dataset shape: {reverse_df.shape}')

amount_features = (GEN - 1) * SIZE * SIZE
X_raw = reverse_df.iloc[:, :amount_features].astype(np.int8)

# Shared index split — same partitioning for all 100 pixels & all 5 architectures
idx            = np.arange(len(X_raw))
idx_trainval, idx_test = train_test_split(idx, test_size=TEST_SIZE,  random_state=RANDOM_STATE)
idx_train,    idx_val  = train_test_split(idx_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE)

# Base 4-D arrays  (N, SIZE, SIZE, TIMESTEPS)
X_base = {}
X_base['train'] = X_raw.iloc[idx_train].to_numpy().reshape(-1, SIZE, SIZE, TIMESTEPS).astype(np.float32)
X_base['val']   = X_raw.iloc[idx_val  ].to_numpy().reshape(-1, SIZE, SIZE, TIMESTEPS).astype(np.float32)
X_base['test']  = X_raw.iloc[idx_test ].to_numpy().reshape(-1, SIZE, SIZE, TIMESTEPS).astype(np.float32)

print(f'Train: {X_base["train"].shape} | Val: {X_base["val"].shape} | Test: {X_base["test"].shape}')

# Ground-truth full boards at t-1  (N_test, 100)
target_cols = [f'Col_{amount_features + i}' for i in range(N_PIXELS)]
y_full       = reverse_df[target_cols].astype(np.int8).to_numpy()
y_test_full  = y_full[idx_test]
print(f'y_test_full shape: {y_test_full.shape}')

In [ ]:
# ─── Add radius features per generation ──────────────────────────────────────
# add_radius_features expects (batch, H, W, C) and uses channel 0 as the binary grid.
# We process each generation independently so every gen gets its own radius counts.
# Result shape: (N, TIMESTEPS, SIZE, SIZE, N_FEAT)

def enrich_with_radius(X_4d: np.ndarray, radii: tuple = RADII,
                       batch_size: int = 2000) -> np.ndarray:
    """
    X_4d : (N, SIZE, SIZE, TIMESTEPS)  — raw binary boards
    Returns (N, TIMESTEPS, SIZE, SIZE, N_FEAT) float32
      where N_FEAT = 1 + len(radii)  (binary + one count per radius)
    """
    N = X_4d.shape[0]
    n_feat = 1 + len(radii)
    out = np.empty((N, TIMESTEPS, SIZE, SIZE, n_feat), dtype=np.float32)

    for g in range(TIMESTEPS):
        # (N, SIZE, SIZE, 1)
        gen_boards = X_4d[:, :, :, g:g+1].astype(np.float32)

        # Process in batches to stay memory-safe
        enriched_batches = []
        for start in range(0, N, batch_size):
            batch = gen_boards[start:start + batch_size]
            enriched_batches.append(
                add_radius_features(batch, radii=radii).numpy()
            )
        enriched = np.concatenate(enriched_batches, axis=0)  # (N, SIZE, SIZE, N_FEAT)
        out[:, g, :, :, :] = enriched

    return out


print('Building radius features for train / val / test ...')
t0 = time.time()
X_rad = {}
for split, X in X_base.items():
    X_rad[split] = enrich_with_radius(X)
    print(f'  {split:5s}: {X_rad[split].shape}  ({time.time()-t0:.1f}s)')

print(f'\nDone in {time.time()-t0:.1f}s')
print(f'Memory: train={X_rad["train"].nbytes/1e6:.1f} MB, '
      f'val={X_rad["val"].nbytes/1e6:.1f} MB, '
      f'test={X_rad["test"].nbytes/1e6:.1f} MB')

---
## 2 · Architecture Registry

All reshape functions operate on `X_rad[split]` of shape `(N, TIMESTEPS, SIZE, SIZE, N_FEAT)`.

| Key    | Architecture                    | Input shape                            |
|--------|---------------------------------|----------------------------------------|
| `mlp`  | MLP (fully-connected baseline)  | `(TIMESTEPS × SIZE² × N_FEAT,)` = 1200 |
| `cnn`  | Lightweight CNN                 | `(SIZE, SIZE, TIMESTEPS × N_FEAT)` = (10,10,12) |
| `rnn`  | Single LSTM                     | `(TIMESTEPS, SIZE² × N_FEAT)` = (3,400) |
| `crnn` | TimeDistributed Conv + BiLSTM   | `(TIMESTEPS, SIZE, SIZE, N_FEAT)` = (3,10,10,4) |
| `rcnn` | ConvLSTM2D                      | `(TIMESTEPS, SIZE, SIZE, N_FEAT)` = (3,10,10,4) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Reshape helpers — all take (N, TIMESTEPS, SIZE, SIZE, N_FEAT)
# ══════════════════════════════════════════════════════════════════════════════

def reshape_mlp(X):   # (N, TIMESTEPS*SIZE*SIZE*N_FEAT)
    return X.reshape(X.shape[0], -1).astype(np.float32)

def reshape_cnn(X):   # (N, SIZE, SIZE, TIMESTEPS*N_FEAT)
    # merge gen and feature dims into the channel axis
    N = X.shape[0]
    return X.transpose(0, 2, 3, 1, 4).reshape(N, SIZE, SIZE, TIMESTEPS * N_FEAT).astype(np.float32)

def reshape_rnn(X):   # (N, TIMESTEPS, SIZE*SIZE*N_FEAT)
    N = X.shape[0]
    return X.reshape(N, TIMESTEPS, SIZE * SIZE * N_FEAT).astype(np.float32)

def reshape_crnn(X):  # (N, TIMESTEPS, SIZE, SIZE, N_FEAT) — already in this shape
    return X.astype(np.float32)

def reshape_rcnn(X):  # same as crnn
    return X.astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# Model builders (lightweight — fast enough to train 100× per architecture)
# ══════════════════════════════════════════════════════════════════════════════

def build_mlp_pixel(input_shape):
    """Dense MLP baseline.  Input is fully flattened."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(64,  activation='relu'),
        tf.keras.layers.Dense(1,   activation='sigmoid'),
    ], name='pixel_mlp')


def build_cnn_pixel(input_shape):
    """Two Conv2D blocks — channels carry all generations + radius features."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_cnn')


def build_rnn_pixel(input_shape):
    """Single LSTM — each timestep includes spatial + radius features, flattened."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.LSTM(32, activation='tanh'),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_rnn')


def build_crnn_pixel(input_shape):
    """TimeDistributed Conv2D (spatial + radius) + BiLSTM (temporal)."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),          # (T, H, W, N_FEAT)
        tf.keras.layers.TimeDistributed(
            tf.keras.layers.Conv2D(8, (3, 3), activation='relu', padding='same')
        ),
        tf.keras.layers.TimeDistributed(tf.keras.layers.MaxPooling2D((2, 2))),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten()),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, activation='tanh')),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_crnn')


def build_rcnn_pixel(input_shape):
    """ConvLSTM2D — processes spatial and temporal information jointly."""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),          # (T, H, W, N_FEAT)
        tf.keras.layers.ConvLSTM2D(
            filters=16, kernel_size=(3, 3),
            activation='relu', padding='same',
            return_sequences=False,
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid'),
    ], name='pixel_rcnn')


# ══════════════════════════════════════════════════════════════════════════════
# Registry
# ══════════════════════════════════════════════════════════════════════════════

ARCH_REGISTRY = {
    'mlp':  {'build_fn': build_mlp_pixel,  'reshape_fn': reshape_mlp,
             'input_shape': (TIMESTEPS * SIZE * SIZE * N_FEAT,)},
    'cnn':  {'build_fn': build_cnn_pixel,  'reshape_fn': reshape_cnn,
             'input_shape': (SIZE, SIZE, TIMESTEPS * N_FEAT)},
    'rnn':  {'build_fn': build_rnn_pixel,  'reshape_fn': reshape_rnn,
             'input_shape': (TIMESTEPS, SIZE * SIZE * N_FEAT)},
    'crnn': {'build_fn': build_crnn_pixel, 'reshape_fn': reshape_crnn,
             'input_shape': (TIMESTEPS, SIZE, SIZE, N_FEAT)},
    'rcnn': {'build_fn': build_rcnn_pixel, 'reshape_fn': reshape_rcnn,
             'input_shape': (TIMESTEPS, SIZE, SIZE, N_FEAT)},
}

# Sanity-check: build each model once and print param counts
print(f'{"Arch":<6}  {"Input shape":<30}  {"Params":>8}')
print('─' * 50)
for key, cfg in ARCH_REGISTRY.items():
    m = cfg['build_fn'](cfg['input_shape'])
    m.compile(optimizer='adam', loss='binary_crossentropy')
    print(f'{key:<6}  {str(cfg["input_shape"]):<30}  {m.count_params():>8,}')
    del m
    tf.keras.backend.clear_session()

---
## PHASE 1 — Train All 5 Architectures (100 models each)

In [ ]:
def get_callbacks(patience=ES_PATIENCE):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=patience,
            restore_best_weights=True, verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=max(1, patience // 2), min_lr=1e-5, verbose=0,
        ),
    ]

def compute_class_weights(y):
    y_flat = y.reshape(-1).astype(int)
    total  = len(y_flat)
    n0, n1 = int((y_flat == 0).sum()), int((y_flat == 1).sum())
    if n0 == 0 or n1 == 0:
        return None
    return {0: total / (2.0 * n0), 1: total / (2.0 * n1)}

In [ ]:
grand_log = {}   # arch_key → list of per-pixel dicts

for arch_key, arch_cfg in ARCH_REGISTRY.items():

    arch_dir = os.path.join(MODEL_ROOT, arch_key)
    os.makedirs(arch_dir, exist_ok=True)

    X_tr = arch_cfg['reshape_fn'](X_rad['train'])
    X_vl = arch_cfg['reshape_fn'](X_rad['val'])
    input_shape = arch_cfg['input_shape']

    print(f'\n{"═"*56}')
    print(f'  {arch_key.upper()} — input {input_shape}')
    print(f'{"═"*56}')

    arch_log = []
    t0_arch  = time.time()

    for pixel_idx in range(N_PIXELS):
        save_path = os.path.join(
            arch_dir,
            f'model_pixel{pixel_idx:03d}_size{SIZE}_gen{GEN}_{arch_key}_radius.keras'
        )

        # Resume support
        if os.path.exists(save_path):
            if pixel_idx % 10 == 0:
                print(f'  [pixel {pixel_idx:03d}] already exists, skipping ...')
            continue

        t0_pixel = time.time()

        target_col = f'Col_{amount_features + pixel_idx}'
        y_all      = reverse_df[target_col].astype(np.float32).to_numpy()
        y_train    = y_all[idx_train].reshape(-1, 1)
        y_val      = y_all[idx_val  ].reshape(-1, 1)

        model = arch_cfg['build_fn'](input_shape)
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        history = model.fit(
            X_tr, y_train,
            validation_data=(X_vl, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks(),
            class_weight=compute_class_weights(y_train),
            verbose=0,
        )
        model.save(save_path)

        best_val_acc  = max(history.history['val_accuracy'])
        best_val_loss = min(history.history['val_loss'])
        epochs_run    = len(history.history['loss'])
        elapsed       = time.time() - t0_pixel

        arch_log.append({
            'pixel_idx':     pixel_idx,
            'epochs_run':    epochs_run,
            'best_val_acc':  best_val_acc,
            'best_val_loss': best_val_loss,
            'time_s':        elapsed,
        })

        if pixel_idx % 10 == 0 or pixel_idx == N_PIXELS - 1:
            print(f'  [{arch_key.upper()}] pixel {pixel_idx+1:3d}/100  '
                  f'val_acc={best_val_acc:.3f}  '
                  f'epochs={epochs_run:2d}  '
                  f'arch_total={time.time()-t0_arch:.0f}s')

        tf.keras.backend.clear_session()

    grand_log[arch_key] = arch_log
    trained = len(arch_log)
    skipped = N_PIXELS - trained
    print(f'\n  ✓ {arch_key.upper()} done — {trained} trained, {skipped} skipped')

print(f'\n{"═"*56}')
print('✓ All 5 architectures complete.')

In [ ]:
# ─── Training summary table ───────────────────────────────────────────────────
summary_rows = []
for arch_key, log in grand_log.items():
    if log:
        accs = [r['best_val_acc'] for r in log]
        summary_rows.append({
            'Architecture': arch_key.upper(),
            'Pixels trained': len(log),
            'Mean val_acc': f'{np.mean(accs):.3f}',
            'Min val_acc':  f'{np.min(accs):.3f}',
            'Max val_acc':  f'{np.max(accs):.3f}',
        })
if summary_rows:
    display(pd.DataFrame(summary_rows).set_index('Architecture'))

# ─── Per-architecture training plots ─────────────────────────────────────────
archs_with_data = {k: v for k, v in grand_log.items() if v}
if archs_with_data:
    fig, axes = plt.subplots(2, len(archs_with_data),
                             figsize=(4.5 * len(archs_with_data), 7),
                             squeeze=False)
    for col, (arch_key, log) in enumerate(archs_with_data.items()):
        log_df = pd.DataFrame(log)
        axes[0, col].bar(log_df['pixel_idx'], log_df['best_val_acc'], width=1.0)
        axes[0, col].axhline(log_df['best_val_acc'].mean(), color='red',
                             linestyle='--',
                             label=f"mean={log_df['best_val_acc'].mean():.3f}")
        axes[0, col].set_title(f'{arch_key.upper()} — Val Acc')
        axes[0, col].set_xlabel('Pixel')
        axes[0, col].set_ylim(0.5, 1.0)
        axes[0, col].legend(fontsize=8)

        axes[1, col].bar(log_df['pixel_idx'], log_df['epochs_run'],
                         width=1.0, color='orange')
        axes[1, col].set_title(f'{arch_key.upper()} — Early-Stop Epochs')
        axes[1, col].set_xlabel('Pixel')

    plt.tight_layout()
    plt.show()

---
## PHASE 2 — Evaluate All 5 Architectures

In [ ]:
def evaluate_full_boards(y_true, y_pred, architecture_name=''):
    """
    Compute full-board reconstruction metrics.
    y_true, y_pred : (N, 100) int
    Returns a metrics dict.
    """
    assert y_true.shape == y_pred.shape
    n_samples, _ = y_true.shape

    pixel_accs      = (y_true == y_pred).mean(axis=0)
    avg_pix_acc     = float(pixel_accs.mean())
    exact_matches   = (y_true == y_pred).all(axis=1)
    exact_board_acc = float(exact_matches.mean())
    n_exact         = int(exact_matches.sum())

    y_tf = y_true.flatten()
    y_pf = y_pred.flatten()
    precision = float(precision_score(y_tf, y_pf, zero_division=0))
    recall    = float(recall_score   (y_tf, y_pf, zero_division=0))
    f1        = float(f1_score       (y_tf, y_pf, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(y_tf, y_pf).ravel()

    sep = '─' * 54
    print(f'\n{sep}')
    print(f'  Architecture : {architecture_name}')
    print(f'  Test samples : {n_samples:,}')
    print(sep)
    print(f'  1. Avg Pixel-wise Accuracy    : {avg_pix_acc:.4f}  ({avg_pix_acc*100:.2f}%)')
    print(f'  2. Exact Board Match Accuracy : {exact_board_acc:.4f}  ({n_exact}/{n_samples})')
    print(f'  3. Alive-class  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}')
    print(f'     TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}')
    print(sep)

    return dict(
        architecture        = architecture_name,
        n_samples           = n_samples,
        avg_pixel_acc       = avg_pix_acc,
        exact_board_acc     = exact_board_acc,
        n_exact_matches     = n_exact,
        precision_alive     = precision,
        recall_alive        = recall,
        f1_alive            = f1,
        tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
        pixel_accs_per_pixel = pixel_accs.tolist(),
    )

In [ ]:
import glob as _glob

all_arch_results = {}

for arch_key, arch_cfg in ARCH_REGISTRY.items():

    arch_dir = os.path.join(MODEL_ROOT, arch_key)

    # ── Check all 100 files exist ─────────────────────────────────────────────
    missing = [
        i for i in range(N_PIXELS)
        if not os.path.exists(
            os.path.join(arch_dir,
                         f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{arch_key}_radius.keras')
        )
    ]
    if missing:
        print(f'[{arch_key.upper():4s}] SKIP — {len(missing)} model(s) missing '
              f'(e.g. pixel {missing[0]:03d})')
        continue

    print(f'\n{"═"*54}')
    print(f'  {arch_key.upper()} — loading 100 models ...')

    # ── Load ──────────────────────────────────────────────────────────────────
    t0 = time.time()
    models = [
        tf.keras.models.load_model(
            os.path.join(arch_dir,
                         f'model_pixel{i:03d}_size{SIZE}_gen{GEN}_{arch_key}_radius.keras'),
            compile=False
        )
        for i in range(N_PIXELS)
    ]
    print(f'  Loaded in {time.time()-t0:.1f}s  |  params/model: {models[0].count_params():,}')

    # ── Reshape test set ──────────────────────────────────────────────────────
    X_te = arch_cfg['reshape_fn'](X_rad['test'])

    # ── Predict ───────────────────────────────────────────────────────────────
    t0 = time.time()
    pred_boards = np.zeros((X_te.shape[0], N_PIXELS), dtype=np.int8)
    for i, model in enumerate(models):
        probs = model.predict(X_te, batch_size=512, verbose=0).flatten()
        pred_boards[:, i] = (probs >= 0.5).astype(np.int8)
        if (i + 1) % 10 == 0 or i == N_PIXELS - 1:
            print(f'  Predicted pixel {i+1:3d}/100', end='\r')
    print(f'  Inference done in {time.time()-t0:.1f}s     ')

    del models
    tf.keras.backend.clear_session()

    # ── Evaluate ──────────────────────────────────────────────────────────────
    results = evaluate_full_boards(
        y_true=y_test_full,
        y_pred=pred_boards,
        architecture_name=f'{arch_key.upper()} + Radius (size={SIZE}, gen={GEN})',
    )
    all_arch_results[arch_key] = results

print(f'\n✓ Evaluated {len(all_arch_results)} / {len(ARCH_REGISTRY)} architecture(s).')

---
## 3 · Summary of Results

In [ ]:
# ─── Comparison table ─────────────────────────────────────────────────────────
if not all_arch_results:
    print('No results yet — run Phase 1 + Phase 2 first.')
else:
    numeric_cols = ['Avg Pixel Acc', 'Exact Board Acc', 'Precision', 'Recall', 'F1']
    comparison_df = pd.DataFrame([
        {
            'Architecture':    r['architecture'],
            'Avg Pixel Acc':   r['avg_pixel_acc'],
            'Exact Board Acc': r['exact_board_acc'],
            'Exact Boards':    f"{r['n_exact_matches']}/{r['n_samples']}",
            'Precision':       r['precision_alive'],
            'Recall':          r['recall_alive'],
            'F1':              r['f1_alive'],
        }
        for r in all_arch_results.values()
    ]).set_index('Architecture')

    display(
        comparison_df.style
            .highlight_max(subset=numeric_cols, color='#c6efce')
            .highlight_min(subset=numeric_cols, color='#ffc7ce')
            .format({c: '{:.4f}' for c in numeric_cols})
            .set_caption(f'Local Experts + Radius Features — size={SIZE}, gen={GEN}')
    )

    # Best architecture per metric
    print('\nBest architecture per metric:')
    for col in numeric_cols:
        best_idx = comparison_df[col].idxmax()
        best_val = comparison_df.loc[best_idx, col]
        print(f'  {col:<20}: {best_idx}  ({best_val:.4f})')

In [ ]:
# ─── Per-pixel accuracy heatmaps (one per architecture) ───────────────────────
if all_arch_results:
    n_archs = len(all_arch_results)
    fig, axes = plt.subplots(1, n_archs, figsize=(5 * n_archs, 4), squeeze=False)

    for col, (arch_key, r) in enumerate(all_arch_results.items()):
        grid = np.array(r['pixel_accs_per_pixel']).reshape(SIZE, SIZE)
        im   = axes[0, col].imshow(grid, vmin=0.5, vmax=1.0, cmap='RdYlGn')
        axes[0, col].set_title(
            f'{arch_key.upper()} + Radius\nmean={r["avg_pixel_acc"]:.3f}', fontsize=9
        )
        axes[0, col].set_xlabel('Col')
        axes[0, col].set_ylabel('Row')
        for row in range(SIZE):
            for c2 in range(SIZE):
                axes[0, col].text(c2, row, f'{grid[row, c2]:.2f}',
                                  ha='center', va='center', fontsize=5, color='black')

    fig.colorbar(im, ax=axes[0], label='Per-pixel accuracy', shrink=0.8)
    plt.suptitle(f'Per-pixel Test Accuracy with Radius Features — size={SIZE}, gen={GEN}',
                 y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Bar chart: key metrics across architectures ──────────────────────────────
if all_arch_results:
    arch_labels = [k.upper() for k in all_arch_results]
    metrics_to_plot = {
        'Avg Pixel Acc':   [r['avg_pixel_acc']   for r in all_arch_results.values()],
        'Exact Board Acc': [r['exact_board_acc'] for r in all_arch_results.values()],
        'F1 (Alive)':      [r['f1_alive']        for r in all_arch_results.values()],
    }

    x       = np.arange(len(arch_labels))
    width   = 0.25
    offsets = [-width, 0, width]
    colors  = ['#4472C4', '#ED7D31', '#70AD47']

    fig, ax = plt.subplots(figsize=(max(8, 2 * len(arch_labels)), 5))
    for (label, vals), offset, color in zip(metrics_to_plot.items(), offsets, colors):
        bars = ax.bar(x + offset, vals, width, label=label, color=color, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(arch_labels)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(f'Architecture Comparison — Radius Features (size={SIZE}, gen={GEN})')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()